# 基于高阶API的矩阵乘算子开发

前面我们学习了Ascend C矩阵编程的相关基础知识，本章将在前文基础上，完整呈现如何基于高阶API实现矩阵乘Ascend C算子开发。

本节学习大纲如下：

- 算子分析
- 算子规格描述
- Matmul算子开发

---

## 1. 环境准备
正式开始学习之前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，同时完成了代码目录的创建。保证能正常导入代码以及使用bisheng编译器，完成Matmul直调样例的开发及编译。

In [ ]:
!mkdir -p Sources/03.03

import os
import subprocess
from pathlib import Path

set_env = os.environ.get("ASCEND_TOOLKIT_HOME", "/usr/local/Ascend/cann") + "/set_env.sh"
if not Path(set_env).exists():
    set_env = "/usr/local/Ascend/cann/set_env.sh"

result = subprocess.run(
    ["bash", "-lc", f"source {set_env} && env"],
    capture_output=True,
    text=True,
    check=True,
)
for line in result.stdout.strip().split("\n"):
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value

print("Environment initialization process completed successfully.")

---

## 2. 算子分析：

### 2.1 明确算子的数学表达式及计算逻辑。

Matmul算子的计算公式为：

C = A * B + Bias，其示意图如下。

<img src="./images/03.03/matmul_matrix_multiplication_diagram.png" alt="Matmul矩阵乘示意图"  width="700px" >

- A、B为源操作数，A为左矩阵，形状为[M, K]；B为右矩阵，形状为[K, N]。
- C为目的操作数，存放矩阵乘结果的矩阵，形状为[M, N]。
- Bias为矩阵乘偏置，形状为[1, N]。对A*B结果矩阵的每一行都采用该Bias进行偏置。

使用矩阵乘高阶API编程的计算逻辑是数据需经历“搬运-计算-搬运”三个阶段：
- **CopyIn阶段**对应`SetTensorA`、`SetTensorB`、`SetBias`接口。
- **Compute阶段**对应`Iterate`接口。
- **CopyOut阶段**对应`GetTensorC`接口。

### 2.2 明确输入和输出。
- 输入数量：3个，分别为 a、b、bias。
- 输出数量：1个，为 c。
- 数据类型：输入a、b为half类型，bias与输出为float类型。
- 数据形状（Shape）： 输入a Shape为 (M, K)，输入b Shape为 (K, N), 输入bias Shape为 (1, N), 输出c Shape为（M, N）。此处仅以M = 1024， K = 256， N = 640输入数据作为演示示例，在真实业务场景下，可灵活调整输入数据的shape参数，也可直接支持泛化输入形式。
- 数据布局（Format）：ND。

### 2.3 确定核函数名称和参数。
根据输入输出分析，确定核函数参数，定义核函数原型：
- 核函数名称：matmul_custom
- 参数列表：
- a, b, bias: 输入数据在Global Memory上的起始地址。
- c: 输出数据在Global Memory上的起始地址。
- workspace: 临时计算用空间地址
- tiling: Matmul计算用的切分策略

---

## 3. 算子规格描述

通过以上分析，得到Ascend C Matmul算子的设计规格如下：


<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 550px; border: 1px solid #ddd;">
  <!-- 表头行：算子核函数名 -->
  <tr>
    <td colspan="1" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">算子核函数名</td>
    <td colspan="4" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;text-align: center;" > matmul_custom</td>
  </tr>
  
  <!-- 算子输入模块：纵向合并单元格 -->
  <tr>
    <!-- 算子输入 纵向合并3行（x行+y行+列名行） -->
    <td rowspan="4" style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输入</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">name</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">shape</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">data type</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">format</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">a</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(1024, 256)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">b</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(256, 640)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
   <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">bias</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(640)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">float</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr> 
  <!-- 算子输出模块：纵向合并单元格 -->

  <tr>
  <td  style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输出</td>
    <td style="padding: 8px;">c</td>
    <td style="padding: 8px;">(1024, 640)</td>
    <td style="padding: 8px;">float</td>
    <td style="padding: 8px;">ND</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>  

---

## 4. Matmul算子开发

在[矩阵乘法介绍章节](./03.02_introduction_to_matrix_multiplication.ipynb)介绍了Matmul矩阵乘的数据切分方案和数据流。Ascend C提供一组Matmul高阶API，封装了这些常用的切分和数据搬运、计算的算法逻辑，方便开发者快速实现Matmul矩阵乘法的运算操作。如下图所示，开发者在Host侧通过调用API自动获取Tiling参数，该参数传递到Kernel侧后，在初始化操作时传入，通过几个简单的API即可完成矩阵乘操作。

<img src="./images/03.03/process.png" alt="开发过程"  width="700px" >

本节最终适配以下文件：

- `matmul_custom_tiling.h`：在Host侧计算TCubeTiling参数，用于向Kernel侧传递切分逻辑。
- `matmul_custom.asc`：包含Kernel侧Matmul计算、Host侧Kernel调用和结果验证。
- `CMakeLists.txt`：编译 `.asc` 文件并生成最终可执行产物。

按照如下5个阶段适配算子：
- `Host侧设置Tiling切分逻辑`
- `Kernel实现`
- `Host侧调用Kernel`
- `Host侧验证精度`
- `CMake编译配置`

### 4.1 Host侧Tiling切分逻辑

Matmul 高阶 API 使用 `TCubeTiling` 在主机侧与设备侧之间传递矩阵乘法的切分参数。

`TCubeTiling`是 Matmul 高阶 API 预定义的切分参数结构体，包含 M、N、Ka、Kb、singleCoreM/N/K、baseM/N/K 等核心切分信息。核函数通过该结构体获取完整的矩阵乘法切分配置，用于指导后续的数据切块、搬运及矩阵计算过程。`TCubeTiling`结构体的主要参数说明如下表所示。

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center"><strong>参数名称</strong></td>
  <td align="center"><strong>数据类型</strong></td>
  <td align="center"><strong>说明</strong></td>
</tr>
<tr>
  <td align="left">usedCoreNum</td>
  <td align="left">int</td>
  <td align="left">使用的AI处理器核数，请根据实际情况设置。取值范围为：[1, AI处理器最大核数]。<br><br>该参数与shape相关参数的关系为：usedCoreNum = (M / singleCoreM) * (N / singleCoreN)。<br><br></td>
</tr>
<tr>
  <td align="left">M, N, Ka, Kb</td>
  <td align="left">int</td>
  <td align="left">A、B、C矩阵原始输入的shape大小，以元素为单位。<br><br>M, Ka为A矩阵原始输入的Shape，Kb, N为B矩阵原始输入的Shape。<br><br></td>
</tr>
<tr>
  <td align="left">singleCoreM, singleCoreN, singleCoreK</td>
  <td align="left">int</td>
  <td align="left">A、B、C矩阵单核内shape大小，以元素为单位。该参数取值必须大于0。<br><br>singleCoreK = K，多核处理时不对K进行切分；singleCoreM <= M；singleCoreN <= N。<br><br> </td>
</tr>
<tr>
  <td align="left">baseM, baseN, baseK</td>
  <td align="left">int</td>
  <td align="left">A、B、C矩阵参与一次矩阵乘指令的shape大小，以元素为单位。<br><br>A、B、C矩阵参与一次矩阵乘的shape大小需要按分形对齐。<br><br> </td>
</tr>
<tr>
  <td align="left">depthA1, depthB1</td>
  <td align="left">int</td>
  <td align="left">A1、B1中全载基本块的份数，depthA1为A1中全载baseM * baseK的份数，depthB1为B1中全载baseN * baseK的份数。<br><br></td>
</tr>
<tr>
  <td align="left">stepM, stepN, stepKa, stepKb</td>
  <td align="left">int</td>
  <td align="left">stepM为左矩阵在A1中缓存的buffer M方向上baseM的倍数。<br>stepN为右矩阵在B1中缓存的buffer N方向上baseN的倍数。<br>stepKa为左矩阵在A1中缓存的buffer Ka方向上baseK的倍数。<br>stepKb为右矩阵在B1中缓存的buffer Kb方向上baseK的倍数。<br><br></td>
</tr>
<tr>
  <td align="left">isBias</td>
  <td align="left">int</td>
  <td align="left">是否使能Bias，参数取值如下：<br>0：不使能Bias（默认值）。<br>1：使能Bias。<br></td>
</tr>
<tr>
  <td align="left">iterateOrder</td>
  <td align="left">int</td>
  <td align="left">一次Iterate计算出[baseM, baseN]大小的C矩阵分片，Iterate完成后，Matmul会自动偏移下一次Iterate输出的C矩阵位置，iterOrder表示自动偏移的顺序。参数取值如下：<br>0：先往M轴方向偏移再往N轴方向偏移。<br>1：先往N轴方向偏移再往M轴方向偏移。<br></td>
</tr>
</table>

在 `<<<>>>` 直调样例中，Host侧首先需要生成 Matmul 高阶 API 所需的 Tiling 参数。该逻辑根据输入矩阵形状、数据类型及当前硬件平台特性，动态计算矩阵乘法所需的切分参数（如单核计算尺寸、循环步长、基础块大小等），并填充到 `TCubeTiling` 中，随后作为核函数参数直接传入Kernel侧使用。

因此，生成源码的第一段先放在 `matmul_custom_tiling.h` 中，专门封装Host侧Tiling生成函数。正式写入头文件前，先了解 Ascend C 提供的 Matmul Tiling API 的基本概念及其获取方式。

#### 4.1.1 Matmul Tiling API介绍

Ascend C提供一组Matmul Tiling API，方便用户获取Matmul kernel计算时所需的Tiling参数。用户只需要传入A/B/C矩阵的Position位置、Format格式和DType数据类型等信息，调用API接口，即可获取到TCubeTiling结构体中的相关参数。

Matmul Tiling API分为Matmul单核Tiling接口、多核Tiling接口和BatchMatmul Tiling接口，本章基于多核并行场景，重点介绍多核 Tiling 接口 —— MultiCoreMatmulTiling，其详细说明如下表所示。

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center"><strong>接口</strong></td>
  <td align="center"><strong>功能</strong></td>
</tr>
<tr>
  <td align="left">SetAType</td>
  <td align="left">设置A矩阵的位置，数据格式，数据类型，是否转置等信息。</td>
</tr>
<tr>
  <td align="left">SetBType</td>
  <td align="left">设置B矩阵的位置，数据格式，数据类型，是否转置等信息。</td>
</tr>
<tr>
  <td align="left">SetCType</td>
  <td align="left">设置C矩阵的位置，数据格式，数据类型等信息。</td>
</tr>
<tr>
  <td align="left">SetBiasType</td>
  <td align="left">设置Bias的位置，数据格式，数据类型等信息。</td>
</tr>
<tr>
  <td align="left">SetShape</td>
  <td align="left">设置Matmul单次计算的形状singleM、singleN、singleK，单位为元素个数。</td>
</tr>
<tr>
  <td align="left">SetOrgShape</td>
  <td align="left">设置Matmul计算时的原始完整的形状M、N、Ka、Kb，单位为元素个数。</td>
</tr>
<tr>
  <td align="left">SetBatchInfoForNormal</td>
  <td align="left">设置A/B矩阵的M/N/K轴信息，以及A/B矩阵各自的Batch数。</td>
</tr>
<tr>
  <td align="left">SetFixSplit</td>
  <td align="left">设置固定的baseM、baseN、baseK，单位为元素个数。</td>
</tr>
<tr>
  <td align="left">SetBufferSpace</td>
  <td align="left">设置Matmul计算时可用的L1/L0C/UB空间大小，单位为字节。</td>
</tr>
<tr>
  <td align="left">SetTraverse</td>
  <td align="left">设置遍历方式，M轴优先还是N轴优先。</td>
</tr>
<tr>
  <td align="left">GetBaseM</td>
  <td align="left">获取baseM值。</td>
</tr>
<tr>
  <td align="left">GetBaseN</td>
  <td align="left">获取baseN值。</td>
</tr>
<tr>
  <td align="left">GetBaseK</td>
  <td align="left">获取baseK值。</td>
</tr>
<tr>
  <td align="left">GetTiling</td>
  <td align="left">获取Tiling参数。</td>
</tr>
<tr>
  <td align="left">SetDim</td>
  <td align="left">设置多核Matmul时，可以参与运算的核数。</td>
</tr>
<tr>
  <td align="left">SetSingleRange</td>
  <td align="left">设置singleCoreM/singleCoreN/singleCoreK的最大值与最小值，单位为元素个数。</td>
</tr>
<tr>
  <td align="left">SetSingleShape</td>
  <td align="left">设置Matmul单核计算的形状singleCoreM、singleCoreN、singleCoreK，单位为元素个数。</td>
</tr>
<tr>
  <td align="left">GetSingleShape</td>
  <td align="left">获取计算后的singleCoreM/singleCoreN/singleCoreK。</td>
</tr>
<tr>
  <td align="left">SetAlignSplit</td>
  <td align="left">设置多核切分时singleCoreM/singleCoreN/singleCoreK的对齐值</td>
</tr>
<tr>
  <td align="left">GetCoreNum</td>
  <td align="left">获得多核切分后， 使用的numBlocks。</td>
</tr>
</table>

#### 4.1.2 如何获取Matmul Tiling参数

本节以多核并行场景为例，介绍如何通过 MultiCoreMatmulTiling 接口获取矩阵乘所需的 Tiling 参数，具体流程如下：

1. **创建多核 Tiling 对象**

    实例化 MultiCoreMatmulTiling 对象，并传入平台信息。

2. **配置矩阵属性与切分策略**

    设置 A、B、C、Bias 的数据类型、存储位置及数据格式；同时指定矩阵的 M、N、K 形状信息，并可选配基础分块大小（baseM/baseN）、核数等参数。

3. **生成 Tiling 参数**

    调用 GetTiling 接口，将计算得到的切分参数输出至 TCubeTiling 结构体中，供后续核函数使用。

#### 4.1.3 生成Host侧Tiling头文件

`matmul_custom_tiling.h` 的功能为根据平台信息、矩阵形状和启动核数生成 `TCubeTiling`。

In [ ]:
%%writefile Sources/03.03/matmul_custom_tiling.h

#ifndef MATMUL_CUSTOM_TILING_H
#define MATMUL_CUSTOM_TILING_H

#include <cstdint>
#include <iostream>

#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"

inline bool GenMatmulTiling(platform_ascendc::PlatformAscendC* ascendcPlatform, TCubeTiling &tiling, int32_t m,
    int32_t n, int32_t k, int32_t numBlocks)
{
    matmul_tiling::MultiCoreMatmulTiling cubeTiling(*ascendcPlatform);
    // 配置参与多核并行计算的AI Core数量
    cubeTiling.SetDim(numBlocks);
    // 配置A、B、C、Bias的存储位置、数据格式、数据类型
    cubeTiling.SetAType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND, matmul_tiling::DataType::DT_FLOAT16);
    cubeTiling.SetBType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND, matmul_tiling::DataType::DT_FLOAT16);
    cubeTiling.SetCType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND, matmul_tiling::DataType::DT_FLOAT);
    cubeTiling.SetBiasType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND, matmul_tiling::DataType::DT_FLOAT);

    cubeTiling.SetShape(m, n, k);            // 设置Matmul单次计算的形状
    cubeTiling.SetOrgShape(m, n, k);         // 设置原始完整的形状
    cubeTiling.SetFixSplit(128, 128, -1);    // 固定基础分块尺寸: baseM = 128, baseN = 128, baseK = -1表示由库自动选择
    cubeTiling.SetBias(true);                // 启用Bias计算
    cubeTiling.SetBufferSpace(-1, -1, -1);   // 设置L1缓冲区分配比例，-1表示由库自动选择

    // 调用GetTiling接口，将计算好的切分参数填充到tiling中
    if (cubeTiling.GetTiling(tiling) == -1) {
        std::cerr << "GenMatmulTiling failed." << std::endl;
        return false;
    }
    return true;
}

#endif

### 4.2 Kernel实现

现在编写 `Sources/03.03/matmul_custom.asc` 的Kernel侧实现，同时写入后续Host代码会复用的头文件、命名空间和矩阵规模常量。定义`ASCENDC_CUBE_ONLY` 表示当前样例只使用Cube计算单元。

`matmul_custom` 是 `<<<>>>` 启动的Kernel入口，通过 `__global__ __cube__` 声明运行在设备侧，并且只启动Cube核。入口函数接收A、B、Bias、C、workspace指针，以及Host侧生成的 `AscendC::tiling::TCubeTiling` 结构体。

Kernel侧整体流程如下：

1. 通过 `GlobalTensor` 绑定A、B、Bias、C在GM上的地址和长度。
2. 创建 `Matmul` 高阶API对象，模板参数描述A/B/Bias/C矩阵在Kernel侧的位置、格式和数据类型。
3. 通过 `REGIST_MATMUL_OBJ(&pipe, GetSysWorkSpacePtr(), mm, &tiling)` 注册Matmul对象，并绑定系统workspace和tiling参数。
4. 调用 `mm.SetOrgShape` 设置原始矩阵形状，再通过 `SetTensorA`、`SetTensorB`、`SetBias` 传入输入Tensor。
5. 调用 `mm.IterateAll(cGlobal)` 执行矩阵乘，最后调用 `mm.End()` 释放Matmul计算资源。


In [ ]:
%%writefile Sources/03.03/matmul_custom.asc

#include <algorithm>
#include <cstdint>
#include <iostream>
#include <iterator>
#include <vector>

#include "acl/acl.h"
#include "kernel_operator.h"
#define ASCENDC_CUBE_ONLY
#include "lib/matmul_intf.h"
#include "matmul_custom_tiling.h"

using namespace matmul;

constexpr int32_t M = 1024;
constexpr int32_t K = 256;
constexpr int32_t N = 640;
constexpr int32_t BLOCK_DIM = 1;

__global__ __cube__ void matmul_custom(__gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* bias, __gm__ uint8_t* c,
    __gm__ uint8_t* workspace, AscendC::tiling::TCubeTiling tiling)
{
    AscendC::TPipe pipe;

    AscendC::GlobalTensor<half> aGlobal;
    AscendC::GlobalTensor<half> bGlobal;
    AscendC::GlobalTensor<float> biasGlobal;
    AscendC::GlobalTensor<float> cGlobal;

    aGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ half*>(a), tiling.M * tiling.Ka);
    bGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ half*>(b), tiling.Kb * tiling.N);
    biasGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(bias), tiling.N);
    cGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(c), tiling.M * tiling.N);

    // Matmul模板参数描述A/B/C矩阵在Kernel侧的位置、格式和数据类型：
    // - A、B、Bias、C都来自GM，格式均为ND；
    // - A/B为half，Bias/C为float；
    // - 这里的配置需要和Host侧SetAType、SetBType、SetCType保持一致。
    Matmul<MatmulType<AscendC::TPosition::GM, CubeFormat::ND, half>,
        MatmulType<AscendC::TPosition::GM, CubeFormat::ND, half>,
        MatmulType<AscendC::TPosition::GM, CubeFormat::ND, float>,
        MatmulType<AscendC::TPosition::GM, CubeFormat::ND, float>> mm;
    // 纯Cube场景下，ASCENDC_CUBE_ONLY会使Matmul对象在Cube侧初始化。
    // Matmul高阶API内部通过GetSysWorkSpacePtr获取系统workspace指针。
    REGIST_MATMUL_OBJ(&pipe, GetSysWorkSpacePtr(), mm, &tiling);
    // SetOrgShape设置原始完整矩阵形状，单位为元素。接口要求在SetTensorA、SetTensorB之前调用。
    mm.SetOrgShape(tiling.M, tiling.N, tiling.Ka, tiling.Kb);

    // 步骤1：将当前核的输入地址传递给 Matmul 对象。
    mm.SetTensorA(aGlobal);
    mm.SetTensorB(bGlobal);
    mm.SetBias(biasGlobal);
    // 步骤2：执行完整的矩阵乘法，结果直接写入 cGlobal 指向的全局内存
    mm.IterateAll(cGlobal);
    // 步骤3：结束矩阵乘操作，释放Matmul计算资源
    mm.End();
}

### 4.3 Host侧调用Kernel

在前面的matmul_custom.asc代码基础上追加Host侧调用<<<>>>逻辑。`kernel_matmul` 负责初始化ACL运行时、获取平台信息、申请系统workspace、生成 `TCubeTiling`、分配Host/Device内存、拷贝A/B/Bias输入，并通过 `matmul_custom<<<BLOCK_DIM, nullptr, stream>>>(..., tiling)` 直接把Tiling结构体传入Kernel开始计算。


In [ ]:
%%writefile -a Sources/03.03/matmul_custom.asc

std::vector<float> kernel_matmul(std::vector<aclFloat16>& a, std::vector<aclFloat16>& b, std::vector<float>& bias)
{
    int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    aclInit(nullptr);
    aclrtSetDevice(deviceId);
    aclrtCreateStream(&stream);

    auto ascendcPlatform = platform_ascendc::PlatformAscendCManager::GetInstance();
    if (ascendcPlatform == nullptr) {
        std::cerr << "ascendcPlatform is nullptr." << std::endl;
        return {};
    }
    size_t userWorkspaceSize = 0;
    size_t systemWorkspaceSize = static_cast<size_t>(ascendcPlatform->GetLibApiWorkSpaceSize());
    // Matmul高阶API内部可能使用系统workspace，Host侧申请后作为Kernel参数传入。
    size_t workspaceSize = userWorkspaceSize + systemWorkspaceSize;
    const size_t aSize = a.size() * sizeof(aclFloat16);
    const size_t bSize = b.size() * sizeof(aclFloat16);
    const size_t biasSize = bias.size() * sizeof(float);
    const size_t cSize = M * N * sizeof(float);

    // 计算Host侧tiling
    TCubeTiling tiling;
    if (!GenMatmulTiling(ascendcPlatform, tiling, M, N, K, BLOCK_DIM)) {
        return {};
    }

    // 分配host侧内存
    uint8_t* cHost;
    aclrtMallocHost((void**)(&cHost), cSize);

    // 分配device侧内存
    uint8_t* aDevice;
    uint8_t* bDevice;
    uint8_t* biasDevice;
    uint8_t* cDevice;
    uint8_t* workspaceDevice;
    aclrtMalloc((void**)&aDevice, aSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&bDevice, bSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&biasDevice, biasSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&cDevice, cSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&workspaceDevice, workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST);

    // 将输入host侧数据拷到device中
    aclrtMemcpy(aDevice, aSize, a.data(), aSize, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(bDevice, bSize, b.data(), bSize, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(biasDevice, biasSize, bias.data(), biasSize, ACL_MEMCPY_HOST_TO_DEVICE);

    // 进行device侧计算
    matmul_custom<<<BLOCK_DIM, nullptr, stream>>>(aDevice, bDevice, biasDevice, cDevice, workspaceDevice, tiling);
    aclrtSynchronizeStream(stream);

    aclrtMemcpy(cHost, cSize, cDevice, cSize, ACL_MEMCPY_DEVICE_TO_HOST);
    std::vector<float> c(reinterpret_cast<float*>(cHost), reinterpret_cast<float*>(cHost + cSize));

    // 释放Host和Device的内存
    aclrtFreeHost(cHost);
    aclrtFree(aDevice);
    aclrtFree(bDevice);
    aclrtFree(biasDevice);
    aclrtFree(cDevice);
    aclrtFree(workspaceDevice);
    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();
    return c;
}

### 4.4 Host侧验证

最后在matmul_custom.asc的末尾加上Host侧精度验证逻辑。`VerifyResult` 打印输出和期望结果的前若干个元素，并通过 `std::equal` 对比完整输出与Golden数据；`main` 构造Host侧输入数据并计算Golden。因为样例中设置A全为1，B全为2，Bias全为0.5，`K=256`，因此期望输出为 `512.5`。


In [ ]:
%%writefile -a Sources/03.03/matmul_custom.asc

uint32_t VerifyResult(std::vector<float> &output, std::vector<float> &golden)
{
    auto printTensor = [](std::vector<float> &tensor, const char *name) {
        constexpr size_t maxPrintSize = 10;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
                  std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };

    printTensor(output, "Output");
    printTensor(golden, "Golden");
    if (output.size() == golden.size() && std::equal(golden.begin(), golden.end(), output.begin())) {
        std::cout << "[Success] Case accuracy is verification passed." << std::endl;
        return 0;
    }
    std::cout << "[Failed] Case accuracy is verification failed!" << std::endl;
    return 1;
}

int32_t main(int32_t argc, char *argv[])
{
    float valueA = 1.0f;
    float valueB = 2.0f;
    float valueBias = 0.5f;
    float valueOutput = 512.5f;
    std::vector<aclFloat16> a(M * K, aclFloatToFloat16(valueA));
    std::vector<aclFloat16> b(K * N, aclFloatToFloat16(valueB));
    std::vector<float> bias(N, valueBias);

    std::vector<float> output = kernel_matmul(a, b, bias);
    std::vector<float> golden(M * N, valueOutput);
    return VerifyResult(output, golden);
}

### 4.5 CMake编译配置
CMake配置方式与先前样例一致。由于本样例在Host侧调用 `MultiCoreMatmulTiling` 生成Matmul Tiling，并且调用`PlatformAscendCManager`获取硬件平台信息，因此还需要额外链接 `tiling_api`、`register`、`platform` 等CANN库。


In [ ]:
%%writefile Sources/03.03/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
set(CMAKE_ASC_ARCHITECTURES "dav-2201" CACHE STRING "NPU architecture: dav-2201, dav-3510")
find_package(ASC REQUIRED)
project(matmul_examples LANGUAGES ASC CXX)

add_executable(demo
    matmul_custom.asc
)

target_link_libraries(demo PRIVATE
    tiling_api
    register
    platform
    m
    dl
)

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)

---

## 5. 编译和运行

执行以下命令编译可执行文件：


In [ ]:
!cd Sources/03.03 && \
mkdir -p build && \
cd build && \
cmake -DCMAKE_ASC_ARCHITECTURES=dav-2201 .. && \
make -j

编译成功后，运行可执行，检查输出是否与预期一致：
```cpp
Output: 512.5 512.5 512.5 512.5 512.5 512.5 512.5 512.5 512.5 512.5 ...
Golden: 512.5 512.5 512.5 512.5 512.5 512.5 512.5 512.5 512.5 512.5 ...
[Success] Case accuracy is verification passed.
```

In [ ]:
!cd Sources/03.03/build/ && ./demo

---

## 6. 课后实践

尝试修改代码，实现Matmul的计算。公式为：

C = A * B + Bias

支持M = 1024, N = 2048, K = 512的half类型输入，float类型输出，规格如下：

<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 550px; border: 1px solid #ddd;">
  <!-- 表头行：算子核函数名 -->
  <tr>
    <td colspan="1" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">算子核函数名</td>
    <td colspan="4" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;text-align: center;" > matmul_custom</td>
  </tr>
  
  <!-- 算子输入模块：纵向合并单元格 -->
  <tr>
    <!-- 算子输入 纵向合并3行（x行+y行+列名行） -->
    <td rowspan="4" style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输入</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">name</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">shape</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">data type</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">format</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">a</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(1024, 512)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">b</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(512, 2048)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
   <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">bias</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(2048)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">float</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr> 
  <!-- 算子输出模块：纵向合并单元格 -->

  <tr>
  <td  style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输出</td>
    <td style="padding: 8px;">c</td>
    <td style="padding: 8px;">(1024, 2048)</td>
    <td style="padding: 8px;">float</td>
    <td style="padding: 8px;">ND</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>  

输入A矩阵全为-1，B矩阵全为2，Bias全为0.5。在matmul_custom.asc和matmul_custom_tiling.h中填充算子实现。预期输出C矩阵全为-1023.5。

In [ ]:
%%writefile Sources/03.03/matmul_custom.asc

In [ ]:
%%writefile Sources/03.03/matmul_custom_tiling.h

执行以下命令编译可执行文件, 并且执行产物验证精度是否符合预期：

In [ ]:
!cd Sources/03.03 && \
mkdir -p build && \
cd build && \
cmake -DCMAKE_ASC_ARCHITECTURES=dav-2201 .. && \
make -j && \
./demo

执行以下代码获取答案。

In [ ]:
!cat ./answer/03.03/matmul_custom.asc

In [ ]:
!cat ./answer/03.03/matmul_custom_tiling.h

---

## 7. 课外知识

如果想具体了解矩阵乘使用了哪些基础API完成端到端的数据搬运和计算，可以课后参看如下文档进行更加深入的学习
- [Matmul基础API编程指南](https://gitcode.com/cann/asc-devkit/blob/9.1.0/docs/guide/%E7%AE%97%E5%AD%90%E5%AE%9E%E8%B7%B5%E5%8F%82%E8%80%83/SIMD%E7%AE%97%E5%AD%90%E5%AE%9E%E7%8E%B0/%E7%9F%A9%E9%98%B5%E7%BC%96%E7%A8%8B%EF%BC%88%E5%9F%BA%E7%A1%80API%EF%BC%89/%E5%88%86%E7%A6%BB%E6%A8%A1%E5%BC%8F.md)
- [Matmul基础API计算样例](https://gitcode.com/cann/asc-devkit/blob/9.1.0/examples/01_simd_cpp_api/00_introduction/02_matrix/matmul_basic_api/README.md)
- [Matmul基础API计算最佳实践](https://gitcode.com/cann/asc-devkit/blob/9.1.0/examples/01_simd_cpp_api/05_best_practices/01_matrix_compute/matmul_basic_api_high_performance/README.md)